Importing libraries

In [ ]:
import requests
import yaml

import pandas as pd
import matplotlib.pyplot as plt

from datetime import date



### Task 1
We start by code to fetch the data from [website] using code curtousy of https://frost.met.no/python_example.html

In [ ]:
# Insert your own client ID here from a yaml file containing client_ID: <YOUR CLIENT ID>
with open("key.yaml", "r") as f:
    config = yaml.safe_load(f)


# making a function to request data from frost.met.no
def get_data(requested_data: str, referencetime:str ) -> dict:
    """
    Requests a datatype for the weather station SN17850 in ås for a given referencetime
    from https://frost.met.no/observations/v0.jsonld.

    ARGS:
        requested_data: requested_data: the data that will be requested. Full list of
        available data elements that can be requested at https://frost.met.no/elementtable.

        referencetime: the referencetime for the requested data given as dates given in the format
        2025-01-01/2025-12-31 for data from January 1st 2025 to December 31st 2025.
    Returns:
        a requested data in the form of a dictionary
    """
    client_id = config.get("client_ID")

    endpoint = 'https://frost.met.no/observations/v0.jsonld'
    parameters = {
    'sources': 'SN17850',
    'elements': requested_data,
    'referencetime': referencetime,
    }
    # Issue an HTTP GET request
    r_mean = requests.get(endpoint, parameters, auth=(client_id, ''))
    # Extract JSON data
    json = r_mean.json()

    # verifies if data is retrived
    if r_mean.status_code == 200:
        data = json['data']
        print('Data retrieved from frost.met.no!')
    else:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json['error']['message'])
        print('Reason: %s' % json['error']['reason'])

    return data


# get data for mean air temperature in 2025
data = get_data('mean(air_temperature P1D)', '2025-01-01/2025-12-31')

# This will return a Dataframe with all of the observations in a table format
df_mean = pd.DataFrame()
for i in range(len(data)):
    row = pd.DataFrame(data[i]['observations'])
    row['referenceTime'] = data[i]['referenceTime']
    row['sourceId'] = data[i]['sourceId']
    df_mean = pd.concat([df_mean, row], ignore_index=True)

df_mean = df_mean.reset_index()

# df_mean.head()

Data retrieved from frost.met.no!


,index,elementId,value,unit,level,timeOffset,timeResolution,timeSeriesId,performanceCategory,exposureCategory,qualityCode,referenceTime,sourceId
0,0,mean(air_temperature P1D),-5.2,degC,"{'levelType': 'height_above_ground', 'unit': '...",PT0H,P1D,0,C,1,0.0,2025-01-01T00:00:00.000Z,SN17850:0
1,1,mean(air_temperature P1D),-2.8,degC,"{'levelType': 'height_above_ground', 'unit': '...",PT6H,P1D,0,C,1,NaN,2025-01-01T00:00:00.000Z,SN17850:0
2,2,mean(air_temperature P1D),-11.5,degC,"{'levelType': 'height_above_ground', 'unit': '...",PT0H,P1D,0,C,1,0.0,2025-01-02T00:00:00.000Z,SN17850:0
3,3,mean(air_temperature P1D),-7.2,degC,"{'levelType': 'height_above_ground', 'unit': '...",PT6H,P1D,0,C,1,NaN,2025-01-02T00:00:00.000Z,SN17850:0
4,4,mean(air_temperature P1D),-8.9,degC,"{'levelType': 'height_above_ground', 'unit': '...",PT0H,P1D,0,C,1,0.0,2025-01-03T00:00:00.000Z,SN17850:0


### Task 2

In [ ]:
# code here

### Task 3

In [ ]:
# code here

### Task 4 

In [15]:
# code here
def count_heatwaves(station_id: str, station_name: str, start_and_end_date: str) -> int:
    """Collects data from frost.met.no and count the number of heatwaves in a year.
    Generates a summary yml-file.

    Args:
        station_id: ID of which wather station to collect data from.
        station_name: Name of the wather station.
        start_and_end_date: Start and end date to collect data from (YYYY-MM-DD/YYYY-MM-DD).
    Returns:
        count_heatwaves: Number of heatwaves in the given time period.
    """

    # Collecting max temperature data from frost.net
    parameters_max = {
    'sources': station_id,
    'elements': 'max(air_temperature P1D)',
    'referencetime': start_and_end_date,
    }

    r_max = requests.get(endpoint, parameters_max, auth=(client_id, ''))

    json_max = r_max.json()

    if r_max.status_code == 200:
        data_max = json_max['data']
        print('Data retrieved from frost.met.no!')
    else:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json['error']['message'])
        print('Reason: %s' % json['error']['reason'])

    # Extract dates and max temperatures from list with data
    dates = []
    max_temperatures = []

    for day in data_max:
        date = day['referenceTime']

        for observation in day['observations']:
            if observation['timeOffset'] == 'PT0H':
                max_temperature = observation['value']

                dates.append(date[:10])
                max_temperatures.append(max_temperature)

                break

    # Find all heatwaves and add the affected dates to a list
    heatwaves = []
    start_date = None
    end_date = None
    consecutive_days = 0
    count_heatwaves = 0

    for current_date, temperature in zip(dates, max_temperatures):
        if temperature >= 27:
            if consecutive_days == 0:
                start_date = current_date

            consecutive_days += 1

        else:
            if consecutive_days >= 5:
                heatwaves.append(f'{start_date} - {end_date}')
                count_heatwaves += 1

            consecutive_days = 0
            start_date = None

        end_date = current_date

    # Make a summary, formatted as the example file
    summary = {
        'dates': heatwaves,
        'station_ID': station_id,
        'station_name': station_name,
        'year': start_and_end_date.split('/')[0].split('-')[0],
    }

    # Make a yaml-file with the summary
    with open('heatwave_summary.yml', 'w') as file:
        yaml.dump(summary, file)

    return count_heatwaves


count_heatwaves('SN17850', 'ÅS', '2025-01-01/2025-12-31')

Data retrieved from frost.met.no!


1

### Task 5

In [ ]:
# code here